In [1]:
# 01_construct_validity.ipynb
# Construct validity: does the convergence index separate healthy pluralism (B)
# from oversight failure (C), while Stage-1 dispersion alone does not? Analysis is
# restricted to boards with >=3 respondents, since board-level dispersion is
# otherwise undefined. Figures are greyscale; no captions are embedded.
import numpy as np, pandas as pd
from pathlib import Path
import warnings; warnings.filterwarnings("ignore")
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"savefig.dpi": 600, "font.size": 11,
                     "axes.edgecolor": "black", "axes.linewidth": 0.8})
DATA = Path("..")/"data"; FIG = Path("..")/"results"/"figures"; TAB = Path("..")/"results"/"tables"
FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)
GREY = {"A_consensus": "#bbbbbb", "B_pluralism": "#777777", "C_failure": "#222222"}
MK = {"A_consensus": "o", "B_pluralism": "s", "C_failure": "^"}
LAB = {"A_consensus": "A: consensus", "B_pluralism": "B: healthy pluralism", "C_failure": "C: oversight failure"}
ORDER = ["A_consensus", "B_pluralism", "C_failure"]
def save(fig, name):
    fig.savefig(FIG/f"{name}.png", dpi=600, bbox_inches="tight")
    fig.savefig(FIG/f"{name}.pdf", bbox_inches="tight"); plt.close(fig)

b = pd.read_csv(DATA/"board_analysis.csv")
b = b[b.n_resp >= 3].copy()

# Table 1: metric means (and SD) by ground-truth class
t1 = b.groupby("gt_class")[["disp1","disp2","viol1","viol2","conv_disp","conv_viol"]].agg(["mean","std"]).round(3)
t1.to_csv(TAB/"table01_metrics_by_class.csv")

# Table 2: Cohen's d (B vs C). Dispersion should NOT separate; convergence SHOULD.
def cohend(x, y):
    nx, ny = len(x), len(y)
    sp = np.sqrt(((nx-1)*x.var(ddof=1) + (ny-1)*y.var(ddof=1)) / (nx+ny-2))
    return (x.mean()-y.mean())/sp if sp > 0 else np.nan
B, C = b[b.gt_class == "B_pluralism"], b[b.gt_class == "C_failure"]
mets = ["disp1", "disp2", "conv_disp", "conv_viol", "viol2"]
t2 = pd.DataFrame({"metric": mets, "cohen_d_B_vs_C": [cohend(B[m], C[m]) for m in mets]}).round(3)
t2.to_csv(TAB/"table02_separation_B_vs_C.csv", index=False)

# Figure 1 (key result): Stage-1 dispersion (x) vs convergence index (y), by class
fig, ax = plt.subplots(figsize=(5.2, 4.2))
for c in ORDER:
    s = b[b.gt_class == c]
    ax.scatter(s.disp1, s.conv_disp, c=GREY[c], marker=MK[c],
               edgecolor="black", linewidth=0.5, s=48, label=LAB[c])
ax.set_xlabel("Stage-1 dispersion"); ax.set_ylabel("Convergence index (Stage 1 to Stage 2)")
ax.legend(frameon=True, edgecolor="black")
save(fig, "fig01_dispersion_vs_convergence")

# Figure 2: distributions by class - dispersion (B,C overlap) vs convergence (B,C split)
fig, axes = plt.subplots(1, 2, figsize=(8.4, 4.0))
sns.boxplot(data=b, x="gt_class", y="disp1", order=ORDER, hue="gt_class",
            palette=GREY, legend=False, ax=axes[0], linewidth=0.9, fliersize=2)
sns.boxplot(data=b, x="gt_class", y="conv_disp", order=ORDER, hue="gt_class",
            palette=GREY, legend=False, ax=axes[1], linewidth=0.9, fliersize=2)
axes[0].set_ylabel("Stage-1 dispersion"); axes[1].set_ylabel("Convergence index")
for a in axes:
    a.set_xlabel(""); a.set_xticks(range(len(ORDER))); a.set_xticklabels([c.split("_")[0] for c in ORDER])
save(fig, "fig02_separation_panels")

print("construct-validity boards (n_resp>=3):", len(b))
print(t2.to_string(index=False))
print("figures: fig01, fig02  | tables: table01, table02")


construct-validity boards (n_resp>=3): 103
   metric  cohen_d_B_vs_C
    disp1          -0.059
    disp2          -2.239
conv_disp           2.336
conv_viol           1.536
    viol2          -0.938
figures: fig01, fig02  | tables: table01, table02
